# 070 CASE 040 — Grazing classification: Munkarp (Höör, Skåne)

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TobiasEdman/imintengine/main?labpath=notebooks%2Fsdl3-cases%2F070_CASE_040-Grazing-Munkarp.ipynb)

**Purpose:** Automate EU agricultural-support (CAP) compliance checks — classify declared grazing parcels (LPIS blocks) as actively grazed or not, using a Sentinel-2 timeseries through the growing season.

**Partners:**
- **Jordbruksverket (SJV)** — LPIS (Land Parcel Identification System)
- **Naturvårdsverket** — NMD reference
- **Skogsstyrelsen** — forest boundary
- **RISE** — pib-ml-grazing CNN-biLSTM model

**Licence:** CC0 1.0. Grazing model is MIT (RISE, 2025).

## 1. Method

Pipeline:

```
LPIS polygon  →  fetch Sentinel-2 timeseries (T × 12-band × H × W)
              →  CNN-biLSTM (GrazingAnalyzer)
                  • active_grazing / no_activity / error
              →  cross-reference NMD
```

`GrazingAnalyzer` operates on a *per-polygon timeseries*, not a single-date `IMINTJob`. It's therefore **not in `ANALYZER_REGISTRY`** — it ships as a standalone class that you instantiate and call directly.

**Reference result (Munkarp, 80 LPIS blocks, 19 dates in growing season 2025):**
- 68 actively grazed (85%)
- 8 no activity (10%)
- 4 classification errors (5%)
- Mean confidence: 0.73

## 2. Setup

Real execution of this notebook needs three things that live outside a plain Binder/CI environment:

1. **LPIS polygons** for the AOI (open data from Jordbruksverket).
2. **Sentinel-2 timeseries** for each polygon — best fetched via DES/CDSE credentials.
3. **Model weights** for `pib_grazing` — shipped with ImintEngine under `imint/fm/pib_grazing/`.

This notebook is therefore structured as a **walkthrough of the analyzer's API** plus a minimal numeric smoke-test on synthetic data, followed by links to the live-data path.

In [ ]:
import numpy as np
from pathlib import Path
from imint.analyzers.grazing import GrazingAnalyzer, GrazingPrediction

# AOI metadata (real runs use this)
AOI = {"west": 13.42, "south": 55.935, "east": 13.48, "north": 55.965}
YEAR = 2025
print(f"AOI (Munkarp): {AOI}")
print(f"Reference year: {YEAR}")

## 3. Run analyzer on a synthetic timeseries

Validates that the model loads and predicts without requiring LPIS/DES access.

In [ ]:
class _FakeTS:
    """Stand-in for GrazingTimeseriesResult during smoke-test."""
    def __init__(self, polygon_id, data, dates):
        self.polygon_id = polygon_id
        self.data = data
        self.dates = dates

# 19 dates through growing season, 12 S2 bands, 46×46 pixels (model's crop)
T, C, H, W = 19, 12, 46, 46
rng = np.random.default_rng(42)
synthetic = rng.normal(loc=0.3, scale=0.1, size=(T, C, H, W)).astype(np.float32)
dates = [f"2025-{m:02d}-15" for m in [3, 4, 5, 6, 7, 8, 9, 10][:T]]  # rough
ts = _FakeTS(polygon_id="munkarp-demo-1", data=synthetic, dates=dates)

try:
    analyzer = GrazingAnalyzer()
    pred = analyzer.predict(ts)
    print(f"polygon_id: {pred.polygon_id}")
    print(f"prediction: {pred.class_label} (class={pred.predicted_class})")
    print(f"confidence: {pred.confidence:.3f}")
    print(f"probabilities: {[round(p, 3) for p in pred.probabilities]}")
except FileNotFoundError as exc:
    print(f"Model weights not found: {exc}")
    print("Continue with linked live-data path below.")

## 4. Live-data workflow (reference)

The production pipeline on DES JupyterHub or a local workstation with CDSE credentials looks like this:

```python
from imint.training.tile_fetch import fetch_grazing_timeseries
from imint.analyzers.grazing import GrazingAnalyzer
import geopandas as gpd

# 1. Load LPIS polygons intersecting the AOI
lpis = gpd.read_file("lpis_2025.gpkg").cx[13.42:13.48, 55.935:55.965]

# 2. Fetch timeseries per polygon (~19 dates across growing season)
timeseries_results = [
    fetch_grazing_timeseries(polygon=row.geometry, year=2025, polygon_id=row.block_id)
    for _, row in lpis.iterrows()
]

# 3. Classify all at once
analyzer = GrazingAnalyzer()
predictions = analyzer.predict_batch(timeseries_results)

# 4. Summarise
active = sum(1 for p in predictions if p.predicted_class == 1)
inactive = sum(1 for p in predictions if p.predicted_class == 0)
errors = sum(1 for p in predictions if p.predicted_class == -1)
print(f"Active grazing: {active}, No activity: {inactive}, Errors: {errors}")
```

The live showcase at [digitalearth.se/case](https://digitalearth.se/case/) runs this pipeline nightly.

## 5. Interpretation

**Operational value:**
Jordbruksverket is obliged by EU rules to check declared area-aid. Field inspection covers ≤2% of blocks; satellite-based classification covers 100% so human inspections can be focused on anomalies.

**Caveats:**
- Model is **decision support**, not automated decision-making.
- 5% classification error → stratified field sampling still needed for calibration.
- Early mowing, drought, or flooding can create false "no activity" readings.

**Links to the full pipeline:**
- [pib-ml-grazing (RISE)](https://github.com/aleksispi/pib-ml-grazing) — original model
- [`imint/analyzers/grazing.py`](https://github.com/TobiasEdman/imintengine/blob/main/imint/analyzers/grazing.py)
- [LPIS open data — Jordbruksverket](https://jordbruksverket.se)
- [NMD open data — Naturvårdsverket](https://www.naturvardsverket.se/nmd)